# Mininet actual experiment batch runner

## 현재 연구 구성

이 노트북은 실제 MP4 프레임을 WSL2 Ubuntu Mininet에서 전송해 콘텐츠 중요도와 전송 계층 결정을 결합한 `cross-layer 적응 전송` 정책을 검증한다.

실험 흐름은 원본 비디오 입력, 프레임 메타데이터 및 전송 단위 준비, Stage A 중요도 계산, Stage B 전송 정책 선택, Mininet 실제 TCP/UDP 전송, 실제 송수신 시각과 전송 바이트 기반 지표 계산 순서다.

비교 기준은 `heuristic_frame_aware`를 기존 휴리스틱 baseline, `frame_action_single_path`를 강한 frame-action baseline, `deadline_feasible_frame_action`을 기존 deadline-aware 정책, `gop_aware_deadline_frame_action`을 GOP-aware 개선 정책으로 둔다.

현재 기본 preset인 `gop_aware_focus`는 `archive_popeye_512kb / 1 Mbps / RTT 50 ms / loss 0%` 조건에서 `gop_aware_deadline_frame_action`을 검증한다. 실행 여부, 반복 수, 기존 산출물 초기화 여부는 아래 설정 셀의 `RUN_EXPERIMENTS`, `SKIP_EXISTING`, `REPEAT_COUNT`, `reset_and_new`로 조정한다.


In [9]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
import itertools
import json
import platform
import shlex
import shutil
import subprocess
import time

import pandas as pd

def resolve_repo_root(start_path: Path) -> Path:
    for candidate_path in [start_path, *start_path.parents]:
        if (candidate_path / "scripts" / "mininet_actual_experiment.py").exists():
            return candidate_path.resolve()
    raise RuntimeError(f"Cannot find repository root from: {start_path}")


REPO_ROOT = resolve_repo_root(Path.cwd())

WSL_DISTRIBUTION = "Ubuntu"
REPO_WSL_PATH = "/mnt/c/git/network"
PYTHON_EXECUTABLE_IN_WSL = "python3"
SUDO_PARTS = ["sudo", "-n"]

MATRIX_PRESET = "gop_aware_focus"  # "gop_aware_focus", "fast_1h", "current_focus", "baseline_grid", "full"
RUN_EXPERIMENTS = True
SKIP_EXISTING = False
MAX_JOBS = None
reset_and_new = True

REPEAT_COUNT = 1
PLAYBACK_BUFFER_MS = 50.0
JOB_TIMEOUT_SECONDS = 7200
OUTPUT_ROOT = REPO_ROOT / "output" / "mininet_actual_experiment"
RUN_LOG_DIR = REPO_ROOT / "output" / "mininet_actual_experiment" / "batch_logs"
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)

PRESETS = {
    "gop_aware_focus": {
        "video_names": ["archive_popeye_512kb"],
        "policy_names": ["gop_aware_deadline_frame_action"],
        "bandwidth_values_mbps": [1.0],
        "round_trip_time_values_ms": [50.0],
        "loss_rate_values": [0.0],
    },
    "fast_1h": {
        "video_names": ["archive_popeye_512kb"],
        "policy_names": [
            "heuristic_frame_aware",
            "frame_action_single_path",
            "deadline_feasible_frame_action",
        ],
        "bandwidth_values_mbps": [1.0, 5.0],
        "round_trip_time_values_ms": [10.0, 50.0],
        "loss_rate_values": [0.0],
    },
    "current_focus": {
        "video_names": ["archive_popeye_512kb"],
        "policy_names": ["deadline_feasible_frame_action"],
        "bandwidth_values_mbps": [1.0],
        "round_trip_time_values_ms": [50.0],
        "loss_rate_values": [0.0],
    },
    "baseline_grid": {
        "video_names": ["archive_popeye_512kb"],
        "policy_names": [
            "heuristic_frame_aware",
            "frame_action_single_path",
            "deadline_feasible_frame_action",
        ],
        "bandwidth_values_mbps": [1.0, 2.0, 3.0, 5.0],
        "round_trip_time_values_ms": [10.0],
        "loss_rate_values": [0.0],
    },
    "full": {
        "video_names": [
            "archive_popeye_512kb",
            "echo_mediaelement",
            "w3c_movie_300",
        ],
        "policy_names": [
            "heuristic_frame_aware",
            "frame_action_single_path",
            "deadline_feasible_frame_action",
        ],
        "bandwidth_values_mbps": [1.0, 2.0, 3.0, 5.0],
        "round_trip_time_values_ms": [10.0, 50.0, 100.0],
        "loss_rate_values": [0.0, 1.0, 3.0],
    },
}

if MATRIX_PRESET not in PRESETS:
    raise ValueError(f"Unknown MATRIX_PRESET: {MATRIX_PRESET}")

ACTIVE_MATRIX = PRESETS[MATRIX_PRESET]
print(json.dumps({
    "repo_root": str(REPO_ROOT),
    "wsl_distribution": WSL_DISTRIBUTION,
    "repo_wsl_path": REPO_WSL_PATH,
    "matrix_preset": MATRIX_PRESET,
    "run_experiments": RUN_EXPERIMENTS,
    "skip_existing": SKIP_EXISTING,
    "reset_and_new": reset_and_new,
    "repeat_count": REPEAT_COUNT,
}, indent=2, ensure_ascii=False))


{
  "repo_root": "C:\\git\\network",
  "wsl_distribution": "Ubuntu",
  "repo_wsl_path": "/mnt/c/git/network",
  "matrix_preset": "gop_aware_focus",
  "run_experiments": true,
  "skip_existing": false,
  "reset_and_new": true,
  "repeat_count": 1
}


## Matrix 생성

현재 기본 `gop_aware_focus` preset은 `archive_popeye_512kb`, `gop_aware_deadline_frame_action`, `1 Mbps`, `RTT 50 ms`, `loss 0%` 한 condition만 실행한다. 새 실험을 기존 진행사항 없이 시작하려면 설정 셀에서 `reset_and_new = True`로 바꾼 뒤 실행한다. `fast_1h`와 `full`은 비교/장시간 실행용 preset이다.


In [10]:
@dataclass(frozen=True)
class ExperimentJob:
    video_name: str
    policy_name: str
    bandwidth_mbps: float
    round_trip_time_ms: float
    loss_rate: float


def format_condition_number(value: float) -> str:
    numeric_value = float(value)
    if numeric_value.is_integer():
        return str(int(numeric_value))
    return f"{numeric_value:g}".replace(".", "p")


def build_condition_directory_name(
    bandwidth_mbps: float,
    round_trip_time_ms: float,
    loss_rate: float,
) -> str:
    bandwidth_name = f"{format_condition_number(bandwidth_mbps)}mbps"
    if float(round_trip_time_ms) == 10.0 and float(loss_rate) == 0.0:
        return bandwidth_name
    return (
        f"{bandwidth_name}_"
        f"rtt{format_condition_number(round_trip_time_ms)}ms_"
        f"loss{format_condition_number(loss_rate)}pct"
    )


def output_directory_for(job: ExperimentJob) -> Path:
    return (
        OUTPUT_ROOT
        / job.video_name
        / job.policy_name
        / build_condition_directory_name(
            job.bandwidth_mbps,
            job.round_trip_time_ms,
            job.loss_rate,
        )
    )


def summary_path_for(job: ExperimentJob) -> Path:
    return output_directory_for(job) / "summary.csv"


def build_jobs() -> list[ExperimentJob]:
    preset = ACTIVE_MATRIX
    jobs = [
        ExperimentJob(
            video_name=video_name,
            policy_name=policy_name,
            bandwidth_mbps=float(bandwidth_mbps),
            round_trip_time_ms=float(round_trip_time_ms),
            loss_rate=float(loss_rate),
        )
        for video_name, policy_name, bandwidth_mbps, round_trip_time_ms, loss_rate in itertools.product(
            preset["video_names"],
            preset["policy_names"],
            preset["bandwidth_values_mbps"],
            preset["round_trip_time_values_ms"],
            preset["loss_rate_values"],
        )
    ]
    if MAX_JOBS is not None:
        jobs = jobs[: int(MAX_JOBS)]
    return jobs


def reset_output_directories_for(jobs: list[ExperimentJob]) -> None:
    output_root = OUTPUT_ROOT.resolve()
    for job in jobs:
        output_directory = output_directory_for(job).resolve()
        if output_directory == output_root or output_root not in output_directory.parents:
            raise RuntimeError(f"Refusing to reset outside OUTPUT_ROOT: {output_directory}")
        if output_directory.exists():
            shutil.rmtree(output_directory)
            print(f"reset output directory: {output_directory.relative_to(REPO_ROOT)}")


JOBS = build_jobs()
if reset_and_new:
    reset_output_directories_for(JOBS)

planned_jobs = pd.DataFrame(
    [
        {
            **asdict(job),
            "output_directory": str(output_directory_for(job).relative_to(REPO_ROOT)),
            "summary_exists": summary_path_for(job).exists(),
        }
        for job in JOBS
    ]
)

print(f"condition count: {len(JOBS)}")
print(f"mininet repeat count: {len(JOBS) * REPEAT_COUNT}")
print(planned_jobs.head(30).to_string(index=False))
if len(planned_jobs) > 30:
    print(f"... {len(planned_jobs) - 30} more conditions")


reset output directory: output\mininet_actual_experiment\archive_popeye_512kb\gop_aware_deadline_frame_action\1mbps_rtt50ms_loss0pct
condition count: 1
mininet repeat count: 1
          video_name                     policy_name  bandwidth_mbps  round_trip_time_ms  loss_rate                                                                                             output_directory  summary_exists
archive_popeye_512kb gop_aware_deadline_frame_action             1.0                50.0        0.0 output\mininet_actual_experiment\archive_popeye_512kb\gop_aware_deadline_frame_action\1mbps_rtt50ms_loss0pct           False


## WSL2/Mininet preflight

`RUN_EXPERIMENTS`가 `False`이면 preflight도 dry-run으로 남긴다. 실제 실행 시에는 `sudo -n true`, `ffprobe`, `mn`, `PyAV`, 원본 MP4 존재 여부를 먼저 확인한다.


In [11]:
def run_wsl_bash(command: str, *, timeout: int | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    if platform.system() == "Windows":
        command_parts = ["wsl", "-d", WSL_DISTRIBUTION, "--", "bash", "-lc", command]
    else:
        command_parts = ["bash", "-lc", command]

    completed = subprocess.run(
        command_parts,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if check and completed.returncode != 0:
        print(completed.stdout[-4000:])
        print(completed.stderr[-4000:])
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {command}")
    return completed


def shell_join(parts: list[object]) -> str:
    return " ".join(shlex.quote(str(part)) for part in parts)


def preflight() -> None:
    video_checks = " && ".join(
        f"test -f {shlex.quote(REPO_WSL_PATH + '/dataset/videos/' + video_name + '.mp4')}"
        for video_name in ACTIVE_MATRIX["video_names"]
    )
    pyav_check = 'import av; print("pyav ok")'
    commands = [
        f"cd {shlex.quote(REPO_WSL_PATH)} && test -f scripts/mininet_actual_experiment.py",
        f"cd {shlex.quote(REPO_WSL_PATH)} && {video_checks}",
        "command -v python3",
        "command -v ffprobe",
        "command -v mn",
        f"cd {shlex.quote(REPO_WSL_PATH)} && python3 -c {shlex.quote(pyav_check)}",
        shell_join(SUDO_PARTS + ["true"]),
    ]

    for command in commands:
        result = run_wsl_bash(command, timeout=120, check=False)
        print(f"$ {command}")
        if result.stdout.strip():
            print(result.stdout.strip())
        if result.returncode != 0:
            if result.stderr.strip():
                print(result.stderr.strip())
            raise RuntimeError(f"Preflight failed: {command}")
    print("preflight ok")


if RUN_EXPERIMENTS:
    preflight()
else:
    print("dry-run: RUN_EXPERIMENTS=False, preflight skipped")


$ cd /mnt/c/git/network && test -f scripts/mininet_actual_experiment.py
$ cd /mnt/c/git/network && test -f /mnt/c/git/network/dataset/videos/archive_popeye_512kb.mp4
$ command -v python3
/usr/bin/python3
$ command -v ffprobe
/usr/bin/ffprobe
$ command -v mn
/usr/bin/mn
$ cd /mnt/c/git/network && python3 -c 'import av; print("pyav ok")'
pyav ok
$ sudo -n true
preflight ok


## 전체 matrix 실행

이 셀은 `RUN_EXPERIMENTS = True`일 때만 실제 Mininet을 실행한다. 각 condition은 기존 CLI와 동일하게 WSL2 Ubuntu에서 `sudo -n python3 scripts/mininet_actual_experiment.py ...` 형태로 순차 실행된다.


In [12]:
def build_experiment_command(job: ExperimentJob) -> str:
    command_parts = SUDO_PARTS + [
        PYTHON_EXECUTABLE_IN_WSL,
        "scripts/mininet_actual_experiment.py",
        "--video-name",
        job.video_name,
        "--policy-name",
        job.policy_name,
        "--bandwidth-mbps",
        job.bandwidth_mbps,
        "--round-trip-time-ms",
        job.round_trip_time_ms,
        "--loss-rate",
        job.loss_rate,
        "--playback-buffer-ms",
        PLAYBACK_BUFFER_MS,
        "--repeat-count",
        REPEAT_COUNT,
        "--python-executable",
        PYTHON_EXECUTABLE_IN_WSL,
    ]
    return f"cd {shlex.quote(REPO_WSL_PATH)} && {shell_join(command_parts)}"


def write_run_log(records: list[dict]) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_log_path = RUN_LOG_DIR / f"batch_run_{MATRIX_PRESET}_{timestamp}.csv"
    pd.DataFrame(records).to_csv(run_log_path, index=False, encoding="utf-8-sig")
    return run_log_path


run_records: list[dict] = []

if not RUN_EXPERIMENTS:
    print("dry-run only. Set RUN_EXPERIMENTS=True in the config cell to execute the matrix.")
else:
    for job_index, job in enumerate(JOBS, start=1):
        output_directory = output_directory_for(job)
        summary_path = summary_path_for(job)
        record = {
            **asdict(job),
            "job_index": job_index,
            "job_count": len(JOBS),
            "output_directory": str(output_directory.relative_to(REPO_ROOT)),
            "summary_csv": str(summary_path.relative_to(REPO_ROOT)),
            "status": "pending",
            "return_code": None,
            "elapsed_seconds": None,
        }

        if SKIP_EXISTING and summary_path.exists():
            record["status"] = "skipped_existing"
            run_records.append(record)
            print(f"[{job_index}/{len(JOBS)}] skipped existing: {summary_path}")
            continue

        command = build_experiment_command(job)
        print(f"[{job_index}/{len(JOBS)}] {job.video_name} | {job.policy_name} | {job.bandwidth_mbps}Mbps | RTT {job.round_trip_time_ms}ms | loss {job.loss_rate}%")
        started_at = time.monotonic()
        completed = run_wsl_bash(command, timeout=JOB_TIMEOUT_SECONDS, check=False)
        elapsed_seconds = time.monotonic() - started_at

        record["return_code"] = completed.returncode
        record["elapsed_seconds"] = round(elapsed_seconds, 3)
        record["stdout_tail"] = completed.stdout[-2000:]
        record["stderr_tail"] = completed.stderr[-2000:]
        record["status"] = "completed" if completed.returncode == 0 else "failed"
        run_records.append(record)

        run_log_path = write_run_log(run_records)
        print(f"status={record['status']} elapsed={record['elapsed_seconds']}s log={run_log_path}")
        if completed.stdout.strip():
            print(completed.stdout[-2000:])
        if completed.returncode != 0:
            if completed.stderr.strip():
                print(completed.stderr[-4000:])
            raise RuntimeError(f"Experiment failed. Partial run log: {run_log_path}")

    run_log_path = write_run_log(run_records)
    print(f"batch complete: {run_log_path}")


[1/1] archive_popeye_512kb | gop_aware_deadline_frame_action | 1.0Mbps | RTT 50.0ms | loss 0.0%
status=completed elapsed=434.906s log=C:\git\network\output\mininet_actual_experiment\batch_logs\batch_run_gop_aware_focus_20260518_032910.csv
{
  "video_name": "archive_popeye_512kb",
  "policy_name": "gop_aware_deadline_frame_action",
  "bandwidth_mbps": 1.0,
  "repeat_count": 1,
  "output_directory": "/mnt/c/git/network/output/mininet_actual_experiment/archive_popeye_512kb/gop_aware_deadline_frame_action/1mbps_rtt50ms_loss0pct",
  "summary_csv": "/mnt/c/git/network/output/mininet_actual_experiment/archive_popeye_512kb/gop_aware_deadline_frame_action/1mbps_rtt50ms_loss0pct/summary.csv"
}

batch complete: C:\git\network\output\mininet_actual_experiment\batch_logs\batch_run_gop_aware_focus_20260518_032910.csv


## 결과 수집

현재 matrix에 해당하는 `summary.csv`를 모두 읽어 핵심 지표를 한 표로 모은다. 실행하지 않은 condition은 `missing_summary`로 남는다.


In [13]:
SUMMARY_COLUMNS = [
    "video_name",
    "policy_name",
    "bandwidth_mbps",
    "round_trip_time_ms",
    "loss_rate",
    "frame_count",
    "late_frame_count",
    "late_frame_ratio",
    "keyframe_late_ratio",
    "decodable_gop_ratio",
    "on_time_goodput_ratio",
    "wasted_late_bytes_ratio",
    "dropped_frame_count",
    "reliable_single_count",
    "unreliable_count",
    "drop_count",
    "reliable_single_late_count",
    "unreliable_late_count",
]

summary_rows = []
missing_rows = []

for job in JOBS:
    summary_path = summary_path_for(job)
    if not summary_path.exists():
        missing_rows.append({**asdict(job), "summary_csv": str(summary_path.relative_to(REPO_ROOT))})
        continue
    frame = pd.read_csv(summary_path)
    row = frame.iloc[0].to_dict()
    row["summary_csv"] = str(summary_path.relative_to(REPO_ROOT))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
missing_summary_df = pd.DataFrame(missing_rows)

print(f"loaded summaries: {len(summary_df)}")
print(f"missing summaries: {len(missing_summary_df)}")
if not summary_df.empty:
    existing_columns = [column for column in SUMMARY_COLUMNS if column in summary_df.columns]
    print(summary_df[existing_columns].sort_values(
        ["video_name", "round_trip_time_ms", "loss_rate", "bandwidth_mbps", "policy_name"]
    ).to_string(index=False))

if not missing_summary_df.empty:
    print("missing summary sample:")
    print(missing_summary_df.head(30).to_string(index=False))


loaded summaries: 1
missing summaries: 0
          video_name                     policy_name  bandwidth_mbps  round_trip_time_ms  loss_rate  frame_count  late_frame_count  late_frame_ratio  keyframe_late_ratio  decodable_gop_ratio  on_time_goodput_ratio  wasted_late_bytes_ratio  dropped_frame_count  reliable_single_count  unreliable_count  drop_count  reliable_single_late_count  unreliable_late_count
archive_popeye_512kb gop_aware_deadline_frame_action             1.0                50.0        0.0        11098              6454          0.581546             0.709189              0.28973               0.912192                 0.087808                 6304                    400              4394        6304                         131                     19


## 정책 비교 요약

같은 video/bandwidth/RTT/loss 조건 안에서 정책별 지표를 비교한다.


In [14]:
if summary_df.empty:
    print("No summaries loaded yet.")
else:
    compare_columns = [
        "video_name",
        "bandwidth_mbps",
        "round_trip_time_ms",
        "loss_rate",
        "policy_name",
        "late_frame_ratio",
        "keyframe_late_ratio",
        "decodable_gop_ratio",
        "on_time_goodput_ratio",
        "wasted_late_bytes_ratio",
    ]
    compare_columns = [column for column in compare_columns if column in summary_df.columns]
    comparison_df = summary_df[compare_columns].sort_values(
        ["video_name", "round_trip_time_ms", "loss_rate", "bandwidth_mbps", "policy_name"]
    )
    print(comparison_df.to_string(index=False))

    focus = comparison_df[
        (comparison_df["video_name"] == "archive_popeye_512kb")
        & (comparison_df["bandwidth_mbps"].astype(float) == 1.0)
        & (comparison_df["round_trip_time_ms"].astype(float) == 50.0)
        & (comparison_df["loss_rate"].astype(float) == 0.0)
    ]
    if not focus.empty:
        print("\narchive_popeye_512kb / 1 Mbps / RTT 50 ms / loss 0%:")
        print(focus.to_string(index=False))


          video_name  bandwidth_mbps  round_trip_time_ms  loss_rate                     policy_name  late_frame_ratio  keyframe_late_ratio  decodable_gop_ratio  on_time_goodput_ratio  wasted_late_bytes_ratio
archive_popeye_512kb             1.0                50.0        0.0 gop_aware_deadline_frame_action          0.581546             0.709189              0.28973               0.912192                 0.087808

archive_popeye_512kb / 1 Mbps / RTT 50 ms / loss 0%:
          video_name  bandwidth_mbps  round_trip_time_ms  loss_rate                     policy_name  late_frame_ratio  keyframe_late_ratio  decodable_gop_ratio  on_time_goodput_ratio  wasted_late_bytes_ratio
archive_popeye_512kb             1.0                50.0        0.0 gop_aware_deadline_frame_action          0.581546             0.709189              0.28973               0.912192                 0.087808
